In [1]:
# Ensure local project_event_based imports and modules are on sys.path
import sys
from pathlib import Path
repo_root = Path('..').resolve()  # if running from project_event_based/notebooks, adjust accordingly
# Prefer local event-based src then modules
sys.path.insert(0, str((Path.cwd().parent / 'project_event_based' / 'src').resolve()))
sys.path.insert(0, str((Path.cwd().parent / 'modules').resolve()))
print('sys.path head:', sys.path[:3])

sys.path head: ['/Users/qianyunshen/Desktop/Splendor-6759/project_event_based/modules', '/Users/qianyunshen/Desktop/Splendor-6759/project_event_based/project_event_based/src', '/Users/qianyunshen/anaconda3/envs/splendor_event311/lib/python311.zip']


In [2]:

import os
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('seed set to', SEED)

seed set to 42


In [3]:
pip install shimmy

Note: you may need to restart the kernel to use updated packages.


In [40]:
import yaml
import subprocess
from pathlib import Path
import os
import sys

current_script_path = Path("project_event_based/notebooks").resolve()
root_path = current_script_path.parent
while root_path.name != 'Splendor-6759' and root_path.parent != root_path:
    root_path = root_path.parent
if root_path.name != 'Splendor-6759':
    root_path = current_script_path.parent.parent
orig_cfg = root_path / 'project_event_based' / 'configs' / 'training' / 'ppo_event_based.yaml'
train_script = root_path / 'project_event_based' / 'scripts' / 'train_maskable.py'

if not orig_cfg.exists():
    print('Original config not found:', orig_cfg)
else:
    tmp_cfg = Path.cwd() / 'tmp_event_config.yaml'
    with open(orig_cfg, 'r') as f:
        cfg = yaml.safe_load(f)


    cfg['training']['total_timesteps'] = 2000000
    cfg['training']['tensorboard_log'] = 'logs/tensorboards'
    cfg['ppo']['ent_coef'] = 0.05 
    cfg['ppo']['batch_size'] = 256
    cfg['reward']['event_weights'] = [0.01, 5.0, -0.5, 8.0, 50.0, 0.2, 1.0, 2.0, 5.0]
    cfg['environment']['combine_event_and_score'] = False
    cfg['environment']['opponent'] = 'greedy'

    with open(tmp_cfg, 'w') as f:
        yaml.dump(cfg, f)
    

    env = os.environ.copy()
    env["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
    env["PYTHONPATH"] = f"{root_path}/project_event_based/src:{root_path}/modules:" + env.get("PYTHONPATH", "")

    patch_code = (
        "import numpy as np; "
        "np.bool8 = bool; " 
        "import sys, os; "
        "script_path = sys.argv[1]; "
        "sys.argv = sys.argv[1:]; "
        "exec(compile(open(script_path).read(), script_path, 'exec'), {'__file__': script_path, '__name__': '__main__', 'sys': sys, 'os': os, 'np': np})"
    )


    cmd = [sys.executable, '-c', patch_code, str(train_script), '--config', str(tmp_cfg)]
    
    print(' Start...')
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    
    try:
        for line in p.stdout:
            print(line, end='')
    except KeyboardInterrupt:
        p.terminate()
    
    ret = p.wait()
    print('Training process exited with code', ret)

 Start...
Low-level interception activated: Fake shimmy module created to bypass Discrete(200) conversion error
Using config: /Users/qianyunshen/Desktop/Splendor-6759/project_event_based/notebooks/tmp_event_config.yaml
Run prefix: ppo_event_based_20260328_113654
TensorBoard log dir: /Users/qianyunshen/Desktop/Splendor-6759/project_event_based/logs/tensorboards
Checkpoint dir: /Users/qianyunshen/Desktop/Splendor-6759/project_event_based/logs/checkpoints
Total timesteps: 2000000
Opponent mode: greedy
Using cpu device
Start training run: Console will display episode_reward and event_rates tables synchronously...
Logging to /Users/qianyunshen/Desktop/Splendor-6759/project_event_based/logs/tensorboards/MaskablePPO_6
------------------------------------------
| dynamic_weights/        |              |
|    engine_spike         | 11.9         |
|    is_score_up          | 8            |
| events/                 |              |
|    event_0_rate         | 0.209        |
|    event_1_rate    

In [37]:
import numpy as np
if not hasattr(np, 'bool8'):
    np.bool8 = bool
import os
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'
import subprocess
import sys
from pathlib import Path

model_path = Path('/Users/qianyunshen/Desktop/Splendor-6759/project_event_based/logs/checkpoints/greedy/20260328_001150_1300000_steps.zip')
eval_script = Path('../scripts/evaluate_model.py')

cmd = [
    sys.executable,  
    str(eval_script), 
    '--model', str(model_path), 
    '--n_games', '1000',
    '--opponent', 'greedy' 
]

print('Evaluate:')
print(' '.join(cmd))
print('--------------------------------------------------')

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
try:
    for line in proc.stdout:
        print(line, end='')
except KeyboardInterrupt:
    print("\n Interupt！")
    proc.terminate()
ret = proc.wait()


Evaluate:
/Users/qianyunshen/anaconda3/envs/splendor_event311/bin/python ../scripts/evaluate_model.py --model /Users/qianyunshen/Desktop/Splendor-6759/project_event_based/logs/checkpoints/greedy/20260328_001150_1300000_steps.zip --n_games 1000 --opponent greedy
--------------------------------------------------
Low-level interception activated: Fake shimmy module created to bypass Discrete(200) conversion error
🚀 Start: /Users/qianyunshen/Desktop/Splendor-6759/project_event_based/logs/checkpoints/greedy/20260328_001150_1300000_steps.zip vs greedy
✅ Success: Initialized GreedyAgent with 'value' mode

🏆 Win Rate: 76.60%
📈 Player Score: 14.42
📉 Opponent: 8.07
⚖️ Average Score Difference: 6.35


In [11]:
import numpy as np
if not hasattr(np, 'bool8'):
    np.bool8 = bool
import subprocess
import sys
from pathlib import Path

model_path = Path('../logs/checkpoints/ppo_event_based_20260327_195739_2000000_steps.zip')
eval_script = Path('../scripts/evaluate_model.py')
cmd = [
    sys.executable, 
    str(eval_script), 
    '--model', str(model_path), 
    '--n_games', '800',
    '--opponent', 'random' 
]

print('Evaluate...')
print('--------------------------------------------------')

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
try:
    for line in proc.stdout:
        print(line, end='')
except KeyboardInterrupt:
    proc.terminate()
ret = proc.wait()

Evaluate...
--------------------------------------------------
Low-level interception activated: Fake shimmy module created to bypass Discrete(200) conversion error
🚀 Start: ../logs/checkpoints/ppo_event_based_20260327_195739_2000000_steps.zip vs random

🏆 Win Rate: 88.00%
📈 Player Score: 14.88
📉 Opponent: 5.63
⚖️ Average Score Difference: 9.25


In [139]:
!python ../scripts/arena_alphazero.py

/Users/xinyan/cours/IFT6759/Splendor/splendor-agent/Guo/Splendor-6759/project_event_based/notebooks/../scripts/arena_alphazero.py:429: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
  print(f"\Start( {n_games} games)")
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Low-level interception activated: Fake shimmy module created to bypass Discrete(200) conversion error
 Initiate Splendor battle 
waking AlphaZero brain...
AlphaZero ready
waking RL 

In [148]:
!python ../scripts/arena_battle.py

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Low-level interception activated: Fake shimmy module created to bypass Discrete(200) conversion error
 Arena officially open: New vs Old model deathmatch...
Game 1 Finished! Winner Info: True
Game 2 Finished! Winner Info: True
Game 3 Finished! Winner Info: True
Game 4 Finished! Winner Info: True
Game 5 Finished! Winner Info: True
Game 6 Finished! Winner Info: True
Game 7 Finished! Winner Info: True
Game 8 Finished! Winner Info: True
Game 9 Finished! Winner Info: True
Game 10 Finished! Winner Inf

In [124]:
import os
import re
import sys
import yaml
import subprocess
from copy import deepcopy
from pathlib import Path

# -----------------------------
# 1) Paths and base config
# -----------------------------
repo_root = Path.cwd()
while repo_root.name != 'Splendor-6759' and repo_root.parent != repo_root:
    repo_root = repo_root.parent

if repo_root.name != 'Splendor-6759':
    raise RuntimeError('Cannot locate repo root Splendor-6759')

base_cfg_path = repo_root / 'project_event_based' / 'configs' / 'training' / 'ppo_event_based.yaml'
train_script = repo_root / 'project_event_based' / 'scripts' / 'train_maskable.py'
eval_script = repo_root / 'project_event_based' / 'scripts' / 'evaluate_model.py'
checkpoints_root = repo_root / 'project_event_based' / 'logs' / 'checkpoints'

ablation_cfg_dir = repo_root / 'project_event_based' / 'notebooks' / 'ablation_configs'
ablation_cfg_dir.mkdir(parents=True, exist_ok=True)

print('repo_root =', repo_root)
print('base_cfg  =', base_cfg_path)
print('train     =', train_script)
print('eval      =', eval_script)

# -----------------------------
# 2) Helper functions
# -----------------------------
def deep_update(dst, src):
    for k, v in src.items():
        if isinstance(v, dict) and isinstance(dst.get(k), dict):
            deep_update(dst[k], v)
        else:
            dst[k] = v
    return dst

def run_training(exp_name, overrides):
    with open(base_cfg_path, 'r') as f:
        cfg = yaml.safe_load(f)

    cfg = deep_update(cfg, deepcopy(overrides))

    tmp_cfg = ablation_cfg_dir / f'{exp_name}.yaml'
    with open(tmp_cfg, 'w') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)

    env = os.environ.copy()
    env['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'
    env['PYTHONPATH'] = f"{repo_root}/project_event_based/src:{repo_root}/modules:" + env.get('PYTHONPATH', '')

    cmd = [sys.executable, str(train_script), '--config', str(tmp_cfg), '--run-tag', exp_name]
    print('\n' + '=' * 80)
    print('TRAIN:', exp_name)
    print('CMD  :', ' '.join(cmd))
    print('=' * 80)

    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env
    )

    run_prefix = None
    try:
        for line in proc.stdout:
            print(line, end='')
            m = re.search(r'Run prefix:\s*(\S+)', line)
            if m:
                run_prefix = m.group(1)
    except KeyboardInterrupt:
        proc.terminate()

    ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f'Training failed for {exp_name}, code={ret}')

    # Find latest checkpoint for this run prefix in checkpoints tree.
    if run_prefix:
        pattern = f'{run_prefix}_*_steps.zip'
        candidates = list(checkpoints_root.rglob(pattern))
    else:
        # Fallback if prefix line wasn't captured.
        candidates = list(checkpoints_root.rglob(f'*{exp_name}*_steps.zip'))

    if not candidates:
        raise FileNotFoundError(f'No checkpoint found for {exp_name}')

    model_path = max(candidates, key=lambda p: p.stat().st_mtime)
    print('Selected checkpoint:', model_path)
    return model_path, run_prefix

def run_eval(model_path, n_games=400, opponent='greedy'):
    cmd = [
        sys.executable, str(eval_script),
        '--model', str(model_path),
        '--n_games', str(n_games),
        '--opponent', opponent,
    ]

    print('\nEVAL CMD:', ' '.join(cmd))
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )

    lines = []
    try:
        for line in proc.stdout:
            lines.append(line)
            print(line, end='')
    except KeyboardInterrupt:
        proc.terminate()

    ret = proc.wait()
    text = ''.join(lines)

    # Parse win rate if present in evaluate output.
    win_rate = None
    m = re.search(r'Win rate:\s*([0-9]+\.?[0-9]*)%', text, flags=re.IGNORECASE)
    if m:
        win_rate = float(m.group(1))

    return {
        'ret': ret,
        'win_rate_percent': win_rate,
        'raw_tail': '\n'.join(text.splitlines()[-12:])
    }

# -----------------------------
# 3) Experiment matrix
# -----------------------------
# Notes:
# - A_hybrid_start: combine_event_and_score starts as True from step 0.
# - B_curriculum_default: starts False, then callback switches around step 300k.
# - C_event_only_short: starts False and uses <=250k steps to avoid switch window.
EXPERIMENTS = [
    {
        'name': 'A_hybrid_start',
        'overrides': {
            'environment': {
                'opponent': 'greedy',
                'combine_event_and_score': True,
            },
            'training': {
                'total_timesteps': 1_000_000,
            },
        },
    },
    {
        'name': 'B_curriculum_default',
        'overrides': {
            'environment': {
                'opponent': 'greedy',
                'combine_event_and_score': False,
            },
            'training': {
                'total_timesteps': 1_000_000,
            },
        },
    },
    {
        'name': 'C_event_only_short',
        'overrides': {
            'environment': {
                'opponent': 'greedy',
                'combine_event_and_score': False,
            },
            'training': {
                'total_timesteps': 250_000,
            },
        },
    },
]

print('\nPlanned experiments:')
for exp in EXPERIMENTS:
    print('-', exp['name'])

# -----------------------------
# 4) Run switch
# -----------------------------
RUN_NOW = True # Change to True to launch full A/B/C pipeline

if RUN_NOW:
    results = []
    for exp in EXPERIMENTS:
        name = exp['name']
        model_path, run_prefix = run_training(name, exp['overrides'])
        eval_res = run_eval(model_path, n_games=400, opponent='greedy')

        results.append({
            'exp': name,
            'run_prefix': run_prefix,
            'model_path': str(model_path),
            'win_rate_percent': eval_res['win_rate_percent'],
            'eval_return_code': eval_res['ret'],
        })

    print('\n' + '=' * 80)
    print('A/B/C SUMMARY')
    print('=' * 80)
    for r in results:
        print(
            f"{r['exp']}: win_rate={r['win_rate_percent']}%, "
            f"ret={r['eval_return_code']}, model={r['model_path']}"
        )

    try:
        import pandas as pd
        display(pd.DataFrame(results).sort_values('win_rate_percent', ascending=False))
    except Exception:
        pass
else:
    print('\nTemplate loaded. Set RUN_NOW=True to execute A/B/C experiments.')

repo_root = /Users/xinyan/cours/IFT6759/Splendor/splendor-agent/Guo/Splendor-6759
base_cfg  = /Users/xinyan/cours/IFT6759/Splendor/splendor-agent/Guo/Splendor-6759/project_event_based/configs/training/ppo_event_based.yaml
train     = /Users/xinyan/cours/IFT6759/Splendor/splendor-agent/Guo/Splendor-6759/project_event_based/scripts/train_maskable.py
eval      = /Users/xinyan/cours/IFT6759/Splendor/splendor-agent/Guo/Splendor-6759/project_event_based/scripts/evaluate_model.py

Planned experiments:
- A_hybrid_start
- B_curriculum_default
- C_event_only_short

TRAIN: A_hybrid_start
CMD  : /opt/homebrew/anaconda3/envs/splendor/bin/python3.14 /Users/xinyan/cours/IFT6759/Splendor/splendor-agent/Guo/Splendor-6759/project_event_based/scripts/train_maskable.py --config /Users/xinyan/cours/IFT6759/Splendor/splendor-agent/Guo/Splendor-6759/project_event_based/notebooks/ablation_configs/A_hybrid_start.yaml --run-tag A_hybrid_start
Gym has been unmaintained since 2022 and does not support NumPy 2.0 a

RuntimeError: Training failed for A_hybrid_start, code=-15

In [43]:
# Multi-seed upgrade: 3 seeds per experiment + mean/std/95% CI summary
import math
from copy import deepcopy

# This cell expects the previous template cell to be executed first, so that
# run_training, run_eval, deep_update, and EXPERIMENTS are already defined.

SEEDS = [42, 43, 44]
EVAL_GAMES_PER_SEED = 400
RUN_MULTI_SEED_NOW = True  # Set True to run the full multi-seed benchmark

if not RUN_MULTI_SEED_NOW:
    print('Multi-seed template loaded. Set RUN_MULTI_SEED_NOW=True to execute.')
else:
    all_runs = []

    for exp in EXPERIMENTS:
        base_name = exp['name']
        for seed in SEEDS:
            exp_name = f"{base_name}_s{seed}"
            overrides = deepcopy(exp['overrides'])
            overrides = deep_update(overrides, {'seed': seed})

            model_path, run_prefix = run_training(exp_name, overrides)
            eval_res = run_eval(model_path, n_games=EVAL_GAMES_PER_SEED, opponent='greedy')

            all_runs.append({
                'exp': base_name,
                'seed': seed,
                'run_prefix': run_prefix,
                'model_path': str(model_path),
                'win_rate_percent': eval_res['win_rate_percent'],
                'eval_return_code': eval_res['ret'],
            })

    # Summarize
    try:
        import pandas as pd

        df_runs = pd.DataFrame(all_runs)
        if df_runs.empty:
            print('No runs collected.')
        else:
            print('\n' + '=' * 90)
            print('MULTI-SEED RAW RESULTS')
            print('=' * 90)
            display(df_runs.sort_values(['exp', 'seed']))

            # 95% CI for mean with n=3 uses t_{0.975, df=2} = 4.303
            t_critical = 4.303

            summary = (
                df_runs.groupby('exp', as_index=False)['win_rate_percent']
                .agg(['count', 'mean', 'std'])
                .reset_index()
                .rename(columns={
                    'count': 'n',
                    'mean': 'win_rate_mean_percent',
                    'std': 'win_rate_std_percent',
                })
            )

            summary['win_rate_std_percent'] = summary['win_rate_std_percent'].fillna(0.0)
            summary['ci95_half_width_percent'] = summary.apply(
                lambda r: (t_critical * r['win_rate_std_percent'] / math.sqrt(r['n'])) if r['n'] > 1 else 0.0,
                axis=1,
            )
            summary['ci95_low_percent'] = summary['win_rate_mean_percent'] - summary['ci95_half_width_percent']
            summary['ci95_high_percent'] = summary['win_rate_mean_percent'] + summary['ci95_half_width_percent']

            summary = summary.sort_values('win_rate_mean_percent', ascending=False)

            print('\n' + '=' * 90)
            print('MULTI-SEED SUMMARY (mean/std/95% CI)')
            print('=' * 90)
            display(summary)

            print('\nTop config by mean win rate:')
            top = summary.iloc[0]
            print(
                f"{top['exp']} | mean={top['win_rate_mean_percent']:.2f}% "
                f"std={top['win_rate_std_percent']:.2f} "
                f"95% CI=[{top['ci95_low_percent']:.2f}, {top['ci95_high_percent']:.2f}]"
            )
    except Exception as e:
        print('Pandas summary failed:', e)
        print('Raw results:')
        for row in all_runs:
            print(row)


TRAIN: A_hybrid_start_s42
CMD  : /Users/qianyunshen/anaconda3/envs/splendor_event311/bin/python /Users/qianyunshen/Desktop/Splendor-6759/project_event_based/scripts/train_maskable.py --config /Users/qianyunshen/Desktop/Splendor-6759/project_event_based/notebooks/ablation_configs/A_hybrid_start_s42.yaml --run-tag A_hybrid_start_s42
Low-level interception activated: Fake shimmy module created to bypass Discrete(200) conversion error
Using config: /Users/qianyunshen/Desktop/Splendor-6759/project_event_based/notebooks/ablation_configs/A_hybrid_start_s42.yaml
Run prefix: ppo_event_based_A_hybrid_start_s42_20260328_155415
TensorBoard log dir: /Users/qianyunshen/Desktop/Splendor-6759/project_event_based/logs/tensorboard
Checkpoint dir: /Users/qianyunshen/Desktop/Splendor-6759/project_event_based/logs/checkpoints
Total timesteps: 1000000
Opponent mode: greedy
Using cpu device
Start training run: Console will display episode_reward and event_rates tables synchronously...
Logging to /Users/qian

,exp,seed,run_prefix,model_path,win_rate_percent,eval_return_code
0,A_hybrid_start,42,ppo_event_based_A_hybrid_start_s42_20260328_15...,/Users/qianyunshen/Desktop/Splendor-6759/proje...,82.50,0
1,A_hybrid_start,43,ppo_event_based_A_hybrid_start_s43_20260328_17...,/Users/qianyunshen/Desktop/Splendor-6759/proje...,77.00,0
2,A_hybrid_start,44,ppo_event_based_A_hybrid_start_s44_20260328_18...,/Users/qianyunshen/Desktop/Splendor-6759/proje...,79.00,0
3,B_curriculum_default,42,ppo_event_based_B_curriculum_default_s42_20260...,/Users/qianyunshen/Desktop/Splendor-6759/proje...,79.25,0
4,B_curriculum_default,43,ppo_event_based_B_curriculum_default_s43_20260...,/Users/qianyunshen/Desktop/Splendor-6759/proje...,80.00,0
5,B_curriculum_default,44,ppo_event_based_B_curriculum_default_s44_20260...,/Users/qianyunshen/Desktop/Splendor-6759/proje...,76.75,0
6,C_event_only_short,42,ppo_event_based_C_event_only_short_s42_2026032...,/Users/qianyunshen/Desktop/Splendor-6759/proje...,78.50,0
7,C_event_only_short,43,ppo_event_based_C_event_only_short_s43_2026032...,/Users/qianyunshen/Desktop/Splendor-6759/proje...,79.50,0
8,C_event_only_short,44,ppo_event_based_C_event_only_short_s44_2026032...,/Users/qianyunshen/Desktop/Splendor-6759/proje...,82.25,0



MULTI-SEED SUMMARY (mean/std/95% CI)


,index,exp,n,win_rate_mean_percent,win_rate_std_percent,ci95_half_width_percent,ci95_low_percent,ci95_high_percent
2,2,C_event_only_short,3,80.083333,1.941863,4.824245,75.259088,84.907579
0,0,A_hybrid_start,3,79.500000,2.783882,6.916105,72.583895,86.416105
1,1,B_curriculum_default,3,78.666667,1.701715,4.227635,74.439032,82.894302



Top config by mean win rate:
C_event_only_short | mean=80.08% std=1.94 95% CI=[75.26, 84.91]
